### Task 2.4 — Gradient check: analytical vs. finite differences

#### Setup

In [26]:
import numpy as np
import torch
import torch.nn as nn

# ---------- model (random initialization) ----------------
torch.manual_seed(1)

class ShallowNet(nn.Module):
    def __init__(self, n_hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, n_hidden),
            nn.Tanh(),
            nn.Linear(n_hidden, 1),
        )
    def forward(self, t):
        return self.net(t)

model = ShallowNet(n_hidden=32)

# trying float 64 as float 32 was a bit imprecise (lets hope)
w1 = model.net[0].weight.detach().numpy().ravel().astype(np.float64)   # (l1,)
b1 = model.net[0].bias.detach().numpy().astype(np.float64)             # (l1,)
w2 = model.net[2].weight.detach().numpy().ravel().astype(np.float64)   # (l1,)
b2 = float(model.net[2].bias.detach().numpy()[0])                      # scalar

# ------------ problem setup -------------
m, mu, k = 1, 5, 500
delta = mu / (2*m)
w0 = np.sqrt(k)
w = np.sqrt(w0**2 - delta**2)
phi = np.arctan(-delta / w)
A = 1 / (2*np.cos(phi))

def x_exact(t):
    return np.exp(-delta*t) * 2*A * np.cos(phi + w*t)

t_data = np.linspace(0, 0.5, 12)[1:-1]     # 10 data points in (0, 0.5)
x_data = x_exact(t_data)
t_phys = np.linspace(0, 1, 50)             # collocation points
lam_phys, lam_ic = 1e-4, 1.0
eps = 1e-5

# ------------ network output and time derivatives ---------------
def N(t, w1, b1, w2, b2):
    return np.tanh(np.outer(t, w1) + b1) @ w2 + b2

def dN(t, w1, b1, w2, b2):
    th = np.tanh(np.outer(t, w1) + b1)
    return (1 - th**2) @ (w2 * w1)

def ddN(t, w1, b1, w2, b2):
    th = np.tanh(np.outer(t, w1) + b1)
    return (-2 * th * (1 - th**2)) @ (w2 * w1**2)

# ------------ loss L = L_data + lam_phys L_phys + lam_ic L_ic --------------
def loss(w1, b1, w2, b2):
    p = (w1, b1, w2, b2)
    r = ddN(t_phys, *p) + mu * dN(t_phys, *p) + k * N(t_phys, *p)
    L_data = np.mean((N(t_data, *p) - x_data)**2)
    L_phys = np.mean(r**2)
    L_ic   = (N(0.0, *p)[0] - 1)**2 + dN(0.0, *p)[0]**2
    return L_data + lam_phys * L_phys + lam_ic * L_ic

def rel_error(a, b):
    return np.linalg.norm(a - b) / np.max([np.linalg.norm(b), np.linalg.norm(a)])


#### $\partial\mathcal{L}/\partial W^{(1)}$

In [27]:
# ---------------- dL/dW1: analytical ----------------
def dL_dw1(w1, b1, w2, b2):
    p = (w1, b1, w2, b2)

    def terms(t):
        th = np.tanh(np.outer(t, w1) + b1)     # (T, l1)
        s2 = 1 - th**2                         # sech^2(z_b(t))
        t = t.reshape(-1, 1)                   # (T, 1)
        dN_dw  = w2 * t * s2
        dN1_dw = w2 * s2 * (1 - 2 * t * w1 * th)
        dN2_dw = -2 * w2 * w1 * s2 * (2 * th + t * w1 * (1 - 3 * th**2))
        return dN_dw, dN1_dw, dN2_dw

    # data term
    dN_d, _, _ = terms(t_data)
    grad_data = 2 / len(t_data) * (N(t_data, *p) - x_data) @ dN_d

    # physics term
    dN_p, dN1_p, dN2_p = terms(t_phys)
    r = ddN(t_phys, *p) + mu * dN(t_phys, *p) + k * N(t_phys, *p)
    grad_phys = 2 / len(t_phys) * r @ (dN2_p + mu * dN1_p + k * dN_p)

    # initial-condition term (the (N(0)-1)^2 part is zero)
    grad_ic = 2 * dN(0.0, *p)[0] * w2 * (1 - np.tanh(b1)**2)

    return grad_data + lam_phys * grad_phys + lam_ic * grad_ic

# ---------------- dL/dW1: finite differences ----------------
def fd_dL_dw1(w1, b1, w2, b2):
    grad = np.zeros_like(w1)
    for b in range(len(w1)):
        w_plus  = w1.copy(); w_plus[b]  += eps
        w_minus = w1.copy(); w_minus[b] -= eps
        grad[b] = (loss(w_plus, b1, w2, b2) - loss(w_minus, b1, w2, b2)) / (2 * eps)
    return grad

g_anal = dL_dw1(w1, b1, w2, b2)
g_fd   = fd_dL_dw1(w1, b1, w2, b2)

print("dL/dW1")
print("max abs diff :", np.max(np.abs(g_anal - g_fd)))
print("relative err :", rel_error(g_anal, g_fd))


dL/dW1
max abs diff : 1.6834950100630408e-11
relative err : 1.2234338053338374e-10


#### $\partial\mathcal{L}/\partial W^{(2)}$

In [29]:
# ---------------- dL/dW2: analytical ----------------
def dL_dw2(w1, b1, w2, b2):
    p = (w1, b1, w2, b2)

    def terms(t):
        th = np.tanh(np.outer(t, w1) + b1)     # (T, l1)
        s2 = 1 - th**2                         # sech^2(z_b(t))
        dN_dw  = th
        dN1_dw = w1 * s2
        dN2_dw = -2 * w1**2 * th * s2
        return dN_dw, dN1_dw, dN2_dw

    # data term
    dN_d, _, _ = terms(t_data)
    grad_data = 2 / len(t_data) * (N(t_data, *p) - x_data) @ dN_d

    # physics term
    dN_p, dN1_p, dN2_p = terms(t_phys)
    r = ddN(t_phys, *p) + mu * dN(t_phys, *p) + k * N(t_phys, *p)
    grad_phys = 2 / len(t_phys) * r @ (dN2_p + mu * dN1_p + k * dN_p)

    # initial-condition terms (both are nonzero here)
    grad_ic = (2 * (N(0.0, *p)[0] - 1) * np.tanh(b1)
               + 2 * dN(0.0, *p)[0] * w1 * (1 - np.tanh(b1)**2))

    return grad_data + lam_phys * grad_phys + lam_ic * grad_ic

# ---------------- dL/dW2: finite differences,----------------
def fd_dL_dw2(w1, b1, w2, b2):
    grad = np.zeros_like(w2)
    for b in range(len(w2)):
        w_plus  = w2.copy(); w_plus[b]  += eps
        w_minus = w2.copy(); w_minus[b] -= eps
        grad[b] = (loss(w1, b1, w_plus, b2) - loss(w1, b1, w_minus, b2)) / (2 * eps)
    return grad

g_anal = dL_dw2(w1, b1, w2, b2)
g_fd   = fd_dL_dw2(w1, b1, w2, b2)

print("dL/dW2")
print("max abs diff :", np.max(np.abs(g_anal - g_fd)))
print("relative err :", rel_error(g_anal, g_fd))


dL/dW2
max abs diff : 1.255129333799232e-11
relative err : 7.674295359138392e-12
